# Unified Validation / Prediction-Accuracy - Qwen3-1.7B (parameterized)

Single notebook that replaces the `[Prediction-ACC] ...1K`, `...Full`, and `...Full...celltype_plot_only` variants. Length (1K vs Full), gene truncation, the R-squared summary plot, and the Drive copy are now **parameters** set by the Colab frontend (`colab_frontends/run_validation_colab.ipynb`) via papermill. (`[prediction_test]...` is a different lineage and is intentionally left standalone.)

In [ ]:
# ===== Papermill parameters (defaults reproduce the "1K_len_cleaned" validation variant) =====
REPO_URL = "https://github.com/HangYu8123/SC_Ageing_Prediction.git"
PROJECT_DIR = "/content/SC_Ageing_Prediction"
DATA_SUBDIR = "fine_tune_chunks"
REFRESH_PROJECT = False

# Checkpoint to evaluate
MODEL_ID = "DaisyCuttie/QWEN3-1.7B-EIGHT-ORGANS-EXTENDED-1K"   # Full: "...-FULL-LENGTH"
MODEL_LABEL = "1K Length Model"                                # shown in plot titles

# Tokenization / gene handling
MAX_LENGTH = 1024
TRUNCATE_GENES = True          # True = 1K; Full variants: False
MAX_GENES = 1000

# Eval
PER_DEVICE_EVAL_BS = 128
EVAL_PART = 11
SEED = 42

# Output
OUTPUT_DIR = "/content/qwen3_prediction_acc_outputs"

# Optional extras
PLOT_R2_SUMMARY = False        # the celltype/R2 variant sets this True
COPY_TO_DRIVE = False          # the Full variants copy *.json/*.png to Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/qwen3_prediction_acc_figures"
RELEASE_RUNTIME = False        # never True under papermill


## Session 1 - Runtime and Project Setup

Steps:
1. Mount Google Drive and prepare the Colab workspace.
2. Clone the project repository into `/content`.
3. Install TPU/XLA-compatible dependencies and verify the runtime.


In [ ]:
# Project setup (papermill-safe). Drive is mounted by the frontend; mount here only if needed.
import os, shutil, subprocess

if not os.path.isdir("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped:", repr(exc))

if os.path.isdir("/content"):
    os.chdir("/content")

if REFRESH_PROJECT and os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
if not os.path.exists(PROJECT_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, PROJECT_DIR])

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    print("HF token detected in environment.")

print("Project directory:", PROJECT_DIR)


In [ ]:
import os
import subprocess
import sys

os.environ["PJRT_DEVICE"] = "TPU"
os.environ["XLA_USE_BF16"] = "1"
os.environ.setdefault("PT_XLA_DEBUG", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

PYTORCH_VERSION = "2.6.0"

def pip_install(packages: list[str], extra_args: list[str] | None = None) -> None:
    args = [sys.executable, "-m", "pip", "install"]
    if extra_args:
        args.extend(extra_args)
    args.extend(packages)
    print("Running:", " ".join(args))
    subprocess.check_call(args)

try:
    import torch  # noqa: F401
    import torch_xla  # noqa: F401
    print("torch:", torch.__version__)
    print("torch_xla:", torch_xla.__version__)
except Exception as exc:
    print("Installing torch_xla because import failed:", repr(exc))
    pip_install(
        [
            f"torch=={PYTORCH_VERSION}",
            f"torch_xla[tpu]=={PYTORCH_VERSION}",
        ],
        extra_args=["-q", "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
    )

pip_install(
    [
        "transformers>=4.45.0",
        "datasets>=2.19.0",
        "accelerate>=0.33.0",
        "huggingface_hub>=0.24.0",
        "evaluate>=0.4.2",
        "matplotlib>=3.8.0",
        "scikit-learn>=1.3.0",
    ],
    extra_args=["-q", "-U"],
)

print("Dependency setup complete. Restart the runtime if Colab asks for it.")


In [ ]:
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr

xr.use_spmd()

print("torch:", torch.__version__)
print("torch_xla:", torch_xla.__version__)
print("PJRT device:", os.environ["PJRT_DEVICE"])
print("Global runtime device count:", xr.global_runtime_device_count())
print("Supported XLA devices:", xm.get_xla_supported_devices())
print("Current XLA device:", torch_xla.device())


## Session 2 - Evaluation Configuration

Steps:
1. Define the model checkpoint and output directory.
2. Configure evaluation split names and file prefixes.
3. Set tokenization and inference parameters.


In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import torch

DATA_DIR = Path(PROJECT_DIR) / DATA_SUBDIR
GLOBAL_EVAL_BS = PER_DEVICE_EVAL_BS
DTYPE = torch.bfloat16

ORGAN_PREFIXES = {
    "bladder": "bladder",
    "brain": "brain",
    "bone": "bone-marrow",
    "limb": "limb-muscle",
    "kidney": "kidney",
    "liver": "liver",
    "lung": "lung",
    "heart": "heart",
}

LABEL2ID = {"1m": 0, "3m": 1, "18m": 2, "24m": 3, "30m": 4}
ID2LABEL = {label_id: label for label, label_id in LABEL2ID.items()}
LABEL_SUFFIX = "Answer with exactly one label from {1m, 3m, 18m, 24m, 30m}."

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

print("Evaluation data directory:", DATA_DIR)
print("Model ID:", MODEL_ID, "| MODEL_LABEL:", MODEL_LABEL)
print("Output directory:", OUTPUT_DIR)
print("MAX_LENGTH:", MAX_LENGTH, "| TRUNCATE_GENES:", TRUNCATE_GENES)


## Session 3 - Evaluation Dataset Loading

Steps:
1. Build the overall validation split from all organ files.
2. Load one separate evaluation split per organ.
3. Collect everything into a single `DatasetDict`.


In [ ]:
import glob
import json

from datasets import Dataset, DatasetDict, load_dataset

print("Files found:", len(glob.glob(f"{DATA_DIR}/*.json")))
print("Example files:", sorted(glob.glob(f"{DATA_DIR}/*.json"))[:5])

def build_eval_files() -> dict[str, str]:
    split_to_file = {}
    for split_name, file_prefix in ORGAN_PREFIXES.items():
        split_to_file[split_name] = str(
            DATA_DIR / f"{file_prefix}_cell_data_part_{EVAL_PART}.json"
        )
    return split_to_file

def load_json_dataset(files: list[str] | str) -> Dataset:
    try:
        return load_dataset("json", data_files=files, split="train")
    except Exception as exc:
        print("Falling back to manual JSON loading:", exc)
        rows = []
        file_list = [files] if isinstance(files, str) else files
        for file_path in file_list:
            with open(file_path, "r") as handle:
                rows.extend(json.load(handle))
        return Dataset.from_list(rows)

split_files = build_eval_files()
validation_files = [split_files[name] for name in ORGAN_PREFIXES.keys()]

raw_splits = {
    "validation": load_json_dataset(validation_files),
}

for split_name, file_path in split_files.items():
    raw_splits[split_name] = load_json_dataset(file_path)

raw = DatasetDict(raw_splits)

print("Validation file sample:", validation_files[0])
print(raw)
print("Columns:", raw["validation"].column_names)
print("Example output label:", raw["validation"][0].get("output"))

## Session 4 - Text Preparation and Tokenization

Steps:
1. Convert each example into the evaluation prompt format.
2. Map labels into integer class IDs.
3. Tokenize all evaluation splits with fixed padding for TPU stability.


In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

# Map literal string numbers to actual integers for sorting.
NUM_MAP = {
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20,
}

def sort_and_truncate_genes(cell_input: str, max_genes: int = 1000) -> str:
    if not cell_input.startswith("Genes: "):
        return cell_input

    parts = cell_input.split("\n")
    genes_line = parts[0][len("Genes: "):]

    tokens = genes_line.split()
    gene_pairs = []
    for i in range(0, len(tokens) - 1, 2):
        gene = tokens[i]
        count_str = tokens[i + 1]
        gene_pairs.append((gene, count_str))

    gene_pairs.sort(key=lambda pair: NUM_MAP.get(pair[1].lower(), 0), reverse=True)
    gene_pairs = gene_pairs[:max_genes]

    parts[0] = "Genes: " + " ".join([f"{gene} {count}" for gene, count in gene_pairs])
    return "\n".join(parts)

def format_text(example: dict) -> dict:
    instruction = (example.get("instruction") or "").strip()
    cell_input = (example.get("input") or "").strip()
    label = (example.get("output") or "").strip()

    if label not in LABEL2ID:
        raise ValueError(f"Unexpected label: {label}")

    processed_input = (
        sort_and_truncate_genes(cell_input, max_genes=MAX_GENES)
        if TRUNCATE_GENES else cell_input
    )

    text = f"{instruction}\n\n{processed_input}\n\n{LABEL_SUFFIX}"
    return {"text": text, "label": LABEL2ID[label]}

raw = raw.map(format_text, remove_columns=raw["validation"].column_names)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.bos_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_batch(batch: dict) -> dict:
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    encoded["labels"] = batch["label"]
    return encoded

tokenized = raw.map(tokenize_batch, batched=True, remove_columns=raw["validation"].column_names)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=128)

print(tokenized)
print(tokenized["validation"][0].keys())


## Session 5 - Model and Inference Setup

Steps:
1. Load the trained sequence classification checkpoint.
2. Configure an evaluation-only `Trainer`.
3. Prepare TPU-safe inference settings.


In [ ]:
import inspect

from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model_kwargs = {
    "num_labels": len(LABEL2ID),
    "id2label": ID2LABEL,
    "label2id": LABEL2ID,
    "trust_remote_code": True,
}

pretrained_signature = inspect.signature(AutoModelForSequenceClassification.from_pretrained)
if "dtype" in pretrained_signature.parameters:
    model_kwargs["dtype"] = DTYPE
else:
    model_kwargs["torch_dtype"] = DTYPE

model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, **model_kwargs)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
if getattr(model.config, "bos_token_id", None) is None and tokenizer.bos_token_id is not None:
    model.config.bos_token_id = tokenizer.bos_token_id

eval_arg_candidates = {
    "output_dir": OUTPUT_DIR,
    "per_device_eval_batch_size": GLOBAL_EVAL_BS,
    "bf16": True,
    "dataloader_drop_last": False,
    "dataloader_pin_memory": False,
    "dataloader_num_workers": 0,
    "remove_unused_columns": False,
    "report_to": "none",
    "seed": SEED,
}

valid_eval_args = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
eval_arg_candidates = {
    key: value for key, value in eval_arg_candidates.items() if key in valid_eval_args
}
eval_args = TrainingArguments(**eval_arg_candidates)

trainer_kwargs = {
    "model": model,
    "args": eval_args,
    "data_collator": data_collator,
}

trainer_init_params = set(inspect.signature(Trainer.__init__).parameters.keys())
if "processing_class" in trainer_init_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_init_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer ready for evaluation.")


## Session 6 - Metrics and Plotting Helpers

Steps:
1. Compute confusion matrices and classification reports.
2. Build summary metrics for each split.
3. Save one metrics figure and one confusion matrix per split.


In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches


LABEL_IDS = sorted(ID2LABEL.keys())
LABEL_NAMES = [ID2LABEL[label_id] for label_id in LABEL_IDS]

def get_logits(predictions):
    if isinstance(predictions, (tuple, list)):
        return predictions[0]
    return predictions

def evaluate_split(split_name: str, y_true: np.ndarray, logits: np.ndarray, metrics: dict | None = None) -> dict:
    y_true = np.asarray(y_true).reshape(-1)
    logits = np.asarray(logits)
    y_pred = np.argmax(logits, axis=-1).reshape(-1)

    cm = confusion_matrix(y_true, y_pred, labels=LABEL_IDS)

    per_label_recall = {}
    for row_index, label_id in enumerate(LABEL_IDS):
        denom = cm[row_index, :].sum()
        per_label_recall[ID2LABEL[label_id]] = float(cm[row_index, row_index] / denom) if denom > 0 else float("nan")

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=LABEL_IDS,
        target_names=LABEL_NAMES,
        digits=4,
        output_dict=True,
        zero_division=0,
    )

    derived_metrics = {
        "accuracy": float(report_dict.get("accuracy", float("nan"))),
        "macro_precision": float(report_dict.get("macro avg", {}).get("precision", float("nan"))),
        "macro_recall": float(report_dict.get("macro avg", {}).get("recall", float("nan"))),
        "macro_f1": float(report_dict.get("macro avg", {}).get("f1-score", float("nan"))),
        "weighted_f1": float(report_dict.get("weighted avg", {}).get("f1-score", float("nan"))),
    }

    overall_metrics = dict(metrics or {})
    overall_metrics[f"final_{split_name}_accuracy"] = derived_metrics["accuracy"]
    overall_metrics[f"final_{split_name}_macro_f1"] = derived_metrics["macro_f1"]

    return {
        "split": split_name,
        "num_samples": int(len(y_true)),
        "overall_metrics": overall_metrics,
        "derived_metrics": derived_metrics,
        "per_label_accuracy_rowwise": per_label_recall,
        "confusion_matrix": {"labels": LABEL_NAMES, "matrix": cm.tolist()},
        "classification_report": report_dict,
    }

def plot_accuracy_f1_recall(split_result: dict, output_dir: str) -> None:
    labels = split_result["confusion_matrix"]["labels"]
    report = split_result["classification_report"]

    recalls = []
    f1s = []
    supports = []
    for label in labels:
        recalls.append(split_result["per_label_accuracy_rowwise"].get(label, float("nan")))
        if label in report:
            f1s.append(float(report[label].get("f1-score", 0.0)))
            supports.append(int(report[label].get("support", 0)))
        else:
            f1s.append(0.0)
            supports.append(0)

    recalls_plot = [0.0 if not np.isfinite(value) else float(value) for value in recalls]
    overall_acc = split_result["derived_metrics"]["accuracy"]
    macro_recall = split_result["derived_metrics"]["macro_recall"]
    macro_f1 = split_result["derived_metrics"]["macro_f1"]

    x = np.arange(len(labels))
    bar_width = 0.36
    inner_gap = 0.025

    plt.figure(figsize=(max(10, 0.7 * len(labels)), 6))
    plt.bar(x - (bar_width / 2 + inner_gap / 2), recalls_plot, bar_width, label="Per-class Recall", color="#004567")
    plt.bar(x + (bar_width / 2 + inner_gap / 2), f1s, bar_width, label="Per-class F1", color="#E85A0E")

    if np.isfinite(overall_acc):
        plt.axhline(overall_acc, color="#4D8EC0", linestyle="--", linewidth=2, label=f"Overall Accuracy = {overall_acc:.4f}")

    xticks = [f"{label}\n(n={support})" for label, support in zip(labels, supports)]
    plt.xticks(x, xticks, rotation=45, ha="right")
    plt.ylim(0, 1.05)
    plt.ylabel("Score")
    plt.title(f"[{split_result['split']}] Per-class Recall/F1 + Overall Accuracy {MODEL_LABEL}")

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(True)

    ax.yaxis.grid(True, linestyle="-", linewidth=0.8, alpha=0.5)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", length=0)

    plt.gca().text(
        1.01,
        0.5,
        f"Macro Recall: {macro_recall:.4f}\nMacro F1: {macro_f1:.4f}\nOverall Acc: {overall_acc:.4f}",
        transform=plt.gca().transAxes,
        va="center",
        fontsize=11,
        bbox=dict(boxstyle="round", alpha=0.2),
    )
    plt.legend()
    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    fig_path = os.path.join(output_dir, f"final_{split_result['split']}_metrics.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()
    print("Saved metrics figure to:", fig_path)


def plot_confusion_matrix(split_result: dict, output_dir: str) -> None:
    labels = split_result["confusion_matrix"]["labels"]
    cm = np.array(split_result["confusion_matrix"]["matrix"], dtype=np.int64)

    row_sums = cm.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        cm_norm = np.divide(cm, row_sums, where=(row_sums != 0))
        cm_norm = np.nan_to_num(cm_norm)

    n = len(labels)

    # Separate diagonal and off-diagonal values
    diag_data = np.full_like(cm_norm, np.nan, dtype=float)
    offdiag_data = np.full_like(cm_norm, np.nan, dtype=float)

    for i in range(n):
        for j in range(n):
            if i == j:
                diag_data[i, j] = cm_norm[i, j]
            else:
                offdiag_data[i, j] = cm_norm[i, j]

    # Similar style to your second image
    green_cmap = LinearSegmentedColormap.from_list(
        "agreement_green",
        ["#DDF4DD", "#66C56F", "#006D2C"]
    )

    red_cmap = LinearSegmentedColormap.from_list(
        "soft_difference",
        ["#FAF7F2", "#F2D6C9", "#D98C72"]
    )

    fig, ax = plt.subplots(figsize=(max(7, 0.8 * n), max(6, 0.8 * n)))

    plt.imshow(
        offdiag_data,
        cmap=red_cmap,
        vmin=0,
        vmax=0.25
    )

    plt.imshow(
        diag_data,
        cmap=green_cmap,
        vmin=0.78,
        vmax=0.90
    )

    # Axis labels
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"[{split_result['split']}] Confusion Matrix {MODEL_LABEL}")

    # White grid lines between cells
    ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=2)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Remove outer frame
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Add text
    for i in range(n):
        for j in range(n):
            value = cm_norm[i, j]

            if i == j:
                cell_text = f"{cm[i, j]}\n{cm_norm[i, j]:.2f}"
            else:
                cell_text = f"{cm_norm[i, j]:.2f}"

            plt.text(
                j,
                i,
                cell_text,
                ha="center",
                va="center",
                fontsize=10,
                color="black"
            )

    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    fig_path = os.path.join(output_dir, f"final_{split_result['split']}_confusion_matrix.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved confusion matrix figure to:", fig_path)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, r2_score

ORGAN_DISPLAY = {
    "bladder": "Bladder",
    "brain": "Brain",
    "bone": "Bone-Marrow",
    "limb": "Limb-Muscle",
    "kidney": "Kidney",
    "liver": "Liver",
    "lung": "Lung",
    "heart": "Heart",
}

ORGAN_COLORS = {
    "bladder": "#1f77b4",
    "bone": "#ff7f0e",
    "brain": "#2ca02c",
    "heart": "#d62728",
    "kidney": "#9467bd",
    "limb": "#8c564b",
    "liver": "#e377c2",
    "lung": "#7f7f7f",
    "heart": "#d62728",
}

def plot_organ_r2_summary(results: dict, output_dir: str) -> None:
    organ_order = ["bladder", "bone", "brain", "heart", "kidney", "limb", "liver", "lung"]

    plot_rows = []
    for organ in organ_order:
        if organ not in results:
            continue

        split_result = results[organ]
        report = split_result["classification_report"]

        y_true = []
        y_pred = []

        cm = np.array(split_result["confusion_matrix"]["matrix"], dtype=np.int64)
        labels = split_result["confusion_matrix"]["labels"]

        label_to_id = {label: i for i, label in enumerate(labels)}

        for true_idx, row in enumerate(cm):
            for pred_idx, count in enumerate(row):
                y_true.extend([true_idx] * count)
                y_pred.extend([pred_idx] * count)

        if len(set(y_true)) < 2:
            r2_value = float("nan")
        else:
            r2_value = float(r2_score(y_true, y_pred))

        plot_rows.append(
            {
                "organ": organ,
                "organ_display": ORGAN_DISPLAY.get(organ, organ.title()),
                "r_squared": r2_value,
                "n": split_result["num_samples"],
            }
        )

    plot_rows = [row for row in plot_rows if np.isfinite(row["r_squared"])]

    x_labels = [row["organ_display"] for row in plot_rows]
    y_values = [row["r_squared"] for row in plot_rows]
    bar_colors = [ORGAN_COLORS.get(row["organ"], "#7f7f7f") for row in plot_rows]

    fig, ax = plt.subplots(figsize=(12, 8))
    x = np.arange(len(plot_rows))
    ax.bar(x, y_values, color=bar_colors, width=0.72)

    ax.set_title("Model Performance Summary", fontsize=16, pad=14)
    ax.set_xlabel("Experiments", fontsize=14)
    ax.set_ylabel("Model R-squared Values", fontsize=14)
    ax.set_ylim(0, 1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=12)

    legend_handles = [
        patches.Patch(color=ORGAN_COLORS[organ], label=ORGAN_DISPLAY.get(organ, organ.title()))
        for organ in organ_order
        if organ in results
    ]
    ax.legend(handles=legend_handles, title="Experiment", loc="upper right", frameon=True, fontsize=8, title_fontsize=8)

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)

    fig_path = os.path.join(output_dir, "final_organ_r_squared_summary.png")
    plt.savefig(fig_path, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved organ R-squared summary figure to:", fig_path)

## Session 7 - Run Evaluation Across All Organs

Steps:
1. Run one prediction pass for the overall validation split.
2. Run one prediction pass per organ split.
3. Save JSON summaries, plots, and the evaluated model artifacts.


In [ ]:
results = {}
split_order = ["validation"] + list(ORGAN_PREFIXES.keys())
split_alias = {"validation": "all"}

for split_name in split_order:
    display_name = split_alias.get(split_name, split_name)
    dataset = tokenized[split_name]

    pred_out = trainer.predict(dataset, metric_key_prefix=f"final_{display_name}")
    logits = get_logits(pred_out.predictions)
    y_true = np.asarray(pred_out.label_ids).reshape(-1)
    metrics = getattr(pred_out, "metrics", {}) or {}

    split_result = evaluate_split(display_name, y_true, logits, metrics=metrics)
    results[display_name] = split_result

    print("\n==============================")
    print(f"=== Split: {display_name} (n={split_result['num_samples']}) ===")
    print(
        f"Accuracy: {split_result['derived_metrics']['accuracy']:.6f} | "
        f"Macro-F1: {split_result['derived_metrics']['macro_f1']:.6f}"
    )
    print("==============================")

os.makedirs(OUTPUT_DIR, exist_ok=True)

details_path = os.path.join(OUTPUT_DIR, "final_eval_details_by_split.json")
with open(details_path, "w") as handle:
    json.dump(results, handle, indent=2)

try:
    import torch_xla.core.xla_model as xm
    is_master = xm.is_master_ordinal()
except Exception:
    is_master = True

if is_master:
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Saved model and tokenizer to:", OUTPUT_DIR)

print("\n===== Summary (all + organs) =====")
for split_name in ["all"] + list(ORGAN_PREFIXES.keys()):
    summary = results[split_name]
    print(
        f"{split_name:>10s} | n={summary['num_samples']:6d} | "
        f"accuracy={summary['derived_metrics']['accuracy']:.6f} | "
        f"macro_f1={summary['derived_metrics']['macro_f1']:.6f}"
    )

print("\nSaved detailed evaluation JSON to:", details_path)

In [ ]:
# Optional: reload results from disk if needed
details_path = os.path.join(OUTPUT_DIR, "final_eval_details_by_split.json")

with open(details_path, "r") as handle:
    results = json.load(handle)

for split_name in ["all"] + list(ORGAN_PREFIXES.keys()):
    split_result = results[split_name]

    plot_accuracy_f1_recall(split_result, OUTPUT_DIR)
    plot_confusion_matrix(split_result, OUTPUT_DIR)

In [ ]:
if PLOT_R2_SUMMARY:
    plot_organ_r2_summary(results, OUTPUT_DIR)
else:
    print("PLOT_R2_SUMMARY=False -> skipping organ R-squared summary.")


## Session 8 - Print Per-Label Organ Summaries

Steps:
1. Print label-level recall for each organ split.


In [ ]:
print("\n===== Per-Category Accuracy by Organ =====")
for split_name, split_result in results.items():
    if split_name == "all":
        continue

    print(f"\n--- {split_name.upper()} ---")
    for label, score in split_result["per_label_accuracy_rowwise"].items():
        if np.isfinite(score):
            print(f"  {label}: {score:.4f}")
        else:
            print(f"  {label}: N/A (no samples)")


In [ ]:
# Copy result files (JSON + figures) to Drive when requested (copy, not move, so re-runs work).
if COPY_TO_DRIVE and os.path.isdir("/content/drive/MyDrive"):
    import shutil, glob
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    copied = 0
    for pattern in ("*.json", "*.png"):
        for src in glob.glob(os.path.join(OUTPUT_DIR, pattern)):
            shutil.copy2(src, os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(src)))
            copied += 1
    print(f"Copied {copied} result file(s) to: {DRIVE_OUTPUT_DIR}")
else:
    print("COPY_TO_DRIVE disabled or Drive not mounted; skipping Drive copy.")


## Session 9 - Optional Colab Cleanup

Steps:
1. Release the runtime after evaluation finishes.


In [ ]:
# Optional Colab cleanup. Disabled by default; under papermill this would kill the kernel.
if RELEASE_RUNTIME:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print("runtime.unassign skipped:", repr(exc))
else:
    print("RELEASE_RUNTIME=False -> keeping runtime alive.")
